# 02 – Korrelationen: Erholung, Leistung und Laufmetriken

**Leitfrage 2:** Wie wirken sich Schlaf und HRV auf meine Läufe aus?
**Leitfrage 3:** Wie hängen Pace, Herzfrequenz, Kadenz, Atemfrequenz, Schrittlänge und vertikales Verhältnis zusammen, zwischen Läufen und innerhalb eines Laufs?

Der Baukasten (`explore.py`) arbeitet auf zwei Ebenen:

- **Zwischen Läufen** (`run_table`): eine Zeile pro Lauf mit Mittelwerten über die bewegte Zeit, EF, Wetter, Anstieg, Gewicht (letzte Messung per `join_asof`) und Tageswerten mit Lag.
- **Innerhalb eines Laufs** (`within_run_table`): Messpunkte in 30-Sekunden-Fenstern (`group_by_dynamic`).

**Lag-Konvention:** Lag 0 ist die Nacht direkt vor dem Lauf, also Schlaf, HRV und Ruhepuls des Lauftags. Garmin ordnet die Nacht dem Tag des Aufwachens zu. Werte, die einen ganzen Tag beschreiben (Stress, Schritte, Body Battery), gibt es erst ab Lag 1, dem Vortag, weil der Lauftag den Lauf selbst enthält.

**Lesehilfe:** n = 1, Beobachtungsdaten, viele Tests auf einmal. Alles hier sind **Hypothesen**, keine Belege.

In [ ]:
import matplotlib.pyplot as plt
import polars as pl

from myfitness import explore, plots
from myfitness.explore import Filters

runs = explore.run_table()
windows = explore.within_run_table(Filters(min_moving_s=30 * 60))

print(f"{runs.height} Läufe, davon {runs['ef'].is_not_null().sum()} mit EF; {windows.height} Fenster à 30 s aus Läufen ab 30 min")

## Leitfrage 2: Schlaf und HRV → Leistung

**Zielgröße:** Als Maß für die Tagesleistung nehmen wir `ef_vs_trend`, die Abweichung des EF vom 28-Tage-Mittel (siehe `01_effizienz`). Der rohe EF enthält den Fitness-Trend über das Jahr, und der würde sich mit allem vermischen, was sich ebenfalls mit den Jahreszeiten ändert, auch Schlaf und HRV.

Zuerst die Übersicht: jeder Tageswert bei jedem Lag gegen `ef_vs_trend`.

In [ ]:
daily_values = [
    "sleep_score",
    "sleep_duration_s",
    "sleep_deep_s",
    "hrv_overnight_avg",
    "resting_hr",
    "avg_stress",
    "steps",
    "body_battery_high",
]

lag_overview = pl.concat(
    [
        explore.lag_correlations(runs, value, "ef_vs_trend").with_columns(daily_value=pl.lit(value))
        for value in daily_values
    ]
)
# pivot: one row per daily value, one column per lag
lag_overview.pivot(on="lag", index="daily_value", values="spearman").rename(
    {"0": "Nacht davor", "1": "Vortag", "2": "vorletzter Tag"}
)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 3.8), sharey=True)
for ax, value in zip(axes, ["sleep_score", "hrv_overnight_avg", "resting_hr", "body_battery_high"]):
    plots.plot_lag_correlations(explore.lag_correlations(runs, value, "ef_vs_trend"), value, "ef_vs_trend", ax=ax)
plt.tight_layout()
plt.show()

**Beobachtung:**

- **Nacht vor dem Lauf:** Bei Lag 0 zeigt sich praktisch nichts.
- **Vortag:** Bei Lag 1 weisen dagegen *alle* Erholungswerte in dieselbe Richtung. Besserer Schlaf, höhere HRV, niedrigerer Ruhepuls, weniger Stress und mehr Body Battery am Vortag gehen mit einem etwas besseren EF einher (|ρ| um 0,2–0,26).

**Einordnung:**

- **Zufall:** Bei n ≈ 84 ist |ρ| > 0,21 auch durch Zufall bei etwa 1 von 20 Tests zu erwarten. Hier sind es 21 Tests.
- **Gemeinsame Richtung:** Die Erholungswerte hängen untereinander stark zusammen. Die gemeinsame Richtung ist deshalb weniger unabhängig, als sie aussieht.
- **Mögliche Störgröße:** Gute Erholungswerte am Vortag treten typischerweise nach einem **Ruhetag** auf. Dann ist vielleicht die Pause der eigentliche Grund. Prüfen lässt sich das mit dem Belastungsmodell (M5), etwa über ATL bzw. TSB am Lauftag.

Hypothese für später: *Die Erholung des Vortags sagt die Tagesform besser voraus als die Nacht direkt vor dem Lauf.*

In [ ]:
explore.investigate(runs, "sleep_score_lag1", "ef_vs_trend")

## Leitfrage 3: Laufmetriken zwischen Läufen

Spearman-Korrelationen der Mittelwerte pro Lauf. Spearman misst monotone Zusammenhänge über Ränge und ist robuster gegen Ausreißer als Pearson.

In [ ]:
matrix = explore.correlation_matrix(runs, explore.RUN_METRICS, method="spearman")
plots.plot_correlation_heatmap(matrix, "Laufmetriken zwischen Läufen (Spearman)")
plt.show()

**Lesehilfe:**

- **Tempo als Treiber:** Vieles hängt am Tempo. Schnellere Läufe (niedrigere Pace) haben höhere HF, höhere Kadenz, mehr Oszillation, kürzeren Bodenkontakt und schnellere Atmung. Diese Zusammenhänge sind erwartbar und sagen wenig Neues.
- **Schrittlänge und EF:** Spannender ist der klare Zusammenhang mit ρ ≈ 0,6. Die Kadenz hängt dagegen kaum mit dem EF zusammen. Hypothese: *Effizienter laufe ich, wenn ich längere Schritte mache, nicht schnellere.*

In [ ]:
explore.investigate(runs, "step_length_mm", "ef")

Hält der Zusammenhang, wenn man das Tempo konstant hält? Eine einfache Probe: nur Läufe in **Garmin-Zone 3** (mittlere HF des Laufs).

In [ ]:
zone3_runs = explore.run_table(Filters(hr_zone=3))
explore.investigate(zone3_runs, "step_length_mm", "ef")

Bei der Erstellung (27.10.2026) hielt der Zusammenhang auch in Zone 3 (ρ ≈ 0,61, n = 43). Er ist also nicht nur ein Nebeneffekt unterschiedlicher Belastung.

## Leitfrage 3: innerhalb eines Laufs

Hier geht es darum, wie sich die Größen *während* eines Laufs gemeinsam bewegen. Wer die Fenster aller Läufe zusammenwirft, vermischt das mit Unterschieden *zwischen* Läufen. Deshalb rechnen wir die Korrelation **pro Lauf** und schauen auf die Verteilung über alle Läufe.

In [ ]:
pairs = [
    ("heart_rate", "respiration_rate"),
    ("speed_mps", "cadence_spm"),
    ("speed_mps", "step_length_mm"),
    ("heart_rate", "vertical_ratio_pct"),
]
fig, axes = plt.subplots(2, 2, figsize=(13, 7))
for ax, (x, y) in zip(axes.flat, pairs):
    per_run = explore.within_run_correlations(windows, x, y)
    plots.plot_within_run_distribution(per_run, x, y, ax=ax)
plt.tight_layout()
plt.show()

**Beobachtung:**

- **Atemfrequenz:** Sie folgt der HF in fast jedem Lauf, im Median ρ ≈ 0,64. Die Atemfrequenz vom Handgelenk mit H10-Gurt ist also brauchbar.
- **Tempo:** Wird es schneller, steigen in fast jedem Lauf sowohl Kadenz als auch Schrittlänge. Beide tragen zum Tempo bei.

## Eigenes Paar untersuchen

Eine Zelle genügt. Verfügbare Spalten zeigt `runs.columns` bzw. `windows.columns`.

In [ ]:
explore.investigate(runs, "temp_c", "respiration_rate")

In [ ]:
# Filter combine freely, e.g. only zone-2 runs since July:
# from datetime import date
# explore.run_table(Filters(hr_zone=2, start=date(2026, 7, 1)))
runs.columns

## Grenzen der Aussage

- **n = 1, Beobachtungsdaten:** Korrelation ist keine Ursache. Störgrößen sind Tempo, Temperatur, Anstieg, Trainingsart und Ermüdung.
- **Viele Tests:** Einzelne Treffer um |ρ| ≈ 0,2 sind bei der Anzahl getesteter Paare auch zufällig zu erwarten. Glaubwürdiger sind Muster, die über mehrere Größen hinweg konsistent sind und sich in den nächsten Monaten wiederholen.
- **Laufdynamik vom Handgelenk:** Trends und Zusammenhänge sind aussagekräftig, absolute Werte nur eingeschränkt.
- **Zonen:** Der Filter nutzt Garmins aktuelle Zonen (HF_max 178). Sie gelten rückwirkend für das ganze Jahr.